In [3]:
import csv
import random
import math


def loadcsv(filename):
    lines = csv.reader(open(filename, "r"))

    # Skip the header row
    next(lines)

    dataset = list(lines)

    for i in range(len(dataset)):
        # Convert strings into numbers
        dataset[i] = [float(x) for x in dataset[i]]

    return dataset


def splitdataset(dataset, splitratio):
    # 67% training size
    trainsize = int(len(dataset) * splitratio)

    trainset = []
    copy = list(dataset)

    while len(trainset) < trainsize:
        # Generate random index
        index = random.randrange(len(copy))
        trainset.append(copy.pop(index))

    return [trainset, copy]


def separatebyclass(dataset):
    separated = {}

    # Separate data according to class
    for i in range(len(dataset)):
        vector = dataset[i]

        if vector[-1] not in separated:
            separated[vector[-1]] = []

        separated[vector[-1]].append(vector)

    return separated


def mean(numbers):
    return sum(numbers) / float(len(numbers))


def stdev(numbers):
    avg = mean(numbers)

    variance = sum(
        [pow(x - avg, 2) for x in numbers]
    ) / float(len(numbers) - 1)

    return math.sqrt(variance)


def summarize(dataset):
    # Calculate mean and standard deviation
    summaries = [
        (mean(attribute), stdev(attribute))
        for attribute in zip(*dataset)
    ]

    # Remove class label
    del summaries[-1]

    return summaries


def summarizebyclass(dataset):
    separated = separatebyclass(dataset)

    summaries = {}

    for classvalue, instances in separated.items():

        # Calculate mean and standard deviation
        summaries[classvalue] = summarize(instances)

    return summaries


def calculateprobability(x, mean, stdev):

    exponent = math.exp(
        -(math.pow(x - mean, 2) /
          (2 * math.pow(stdev, 2)))
    )

    return (
        (1 / (math.sqrt(2 * math.pi) * stdev))
        * exponent
    )


def calculateclassprobabilities(summaries, inputvector):

    probabilities = {}

    # Calculate probability for each class
    for classvalue, classsummaries in summaries.items():

        probabilities[classvalue] = 1

        for i in range(len(classsummaries)):

            # Get mean and standard deviation
            mean_value, stdev_value = classsummaries[i]

            # Get test data value
            x = inputvector[i]

            # Calculate probability
            probabilities[classvalue] *= calculateprobability(
                x,
                mean_value,
                stdev_value
            )

    return probabilities


def predict(summaries, inputvector):

    # Calculate class probabilities
    probabilities = calculateclassprobabilities(
        summaries,
        inputvector
    )

    bestLabel = None
    bestProb = -1

    # Select class with highest probability
    for classvalue, probability in probabilities.items():

        if bestLabel is None or probability > bestProb:

            bestProb = probability
            bestLabel = classvalue

    return bestLabel


def getpredictions(summaries, testset):

    predictions = []

    for i in range(len(testset)):

        result = predict(
            summaries,
            testset[i]
        )

        predictions.append(result)

    return predictions


def getaccuracy(testset, predictions):

    correct = 0

    for i in range(len(testset)):

        if testset[i][-1] == predictions[i]:
            correct += 1

    return (correct / float(len(testset))) * 100.0


def main():

    # CSV file name
    filename = '/content/Naive-Bayes-Classification-Data.csv'

    # 67% training and 33% testing
    splitratio = 0.67

    # Load dataset
    dataset = loadcsv(filename)

    # Split dataset
    trainingset, testset = splitdataset(
        dataset,
        splitratio
    )

    print(
        'Split {0} rows into train={1} and test={2} rows'
        .format(
            len(dataset),
            len(trainingset),
            len(testset)
        )
    )

    # Prepare model
    summaries = summarizebyclass(trainingset)

    # Predict test data
    predictions = getpredictions(
        summaries,
        testset
    )

    # Calculate accuracy
    accuracy = getaccuracy(
        testset,
        predictions
    )

    print(
        'Accuracy of the classifier is : {0}%'
        .format(accuracy)
    )


if __name__ == "__main__":
    main()

Split 995 rows into train=666 and test=329 rows
Accuracy of the classifier is : 92.09726443768997%
